# Testing PyTorch compatibility with adaptive sampling

### Cleaned up AI solution

In [2]:
import numpy as np
import torch
import adaptive

# Trainable parameter
theta = torch.tensor(1., dtype=torch.double, requires_grad=True)

def expensive_f_torch(lam: torch.Tensor) -> torch.Tensor:
    return (lam + theta)**2

def expensive_f_numpy(lam: float) -> float:
    """
    Wrapper for adaptive.
    No gradients here by design.
    """
    with torch.no_grad():
        lam_t = torch.tensor(lam, dtype=torch.double)
        return expensive_f_torch(lam_t).item()

# Adaptive sampling
learner = adaptive.Learner1D(expensive_f_numpy, bounds=(0.0, 1.0))
adaptive.runner.simple(learner, loss_goal=0.1)
lambdas = np.array(sorted(learner.data.keys()))
lambdas_torch = torch.tensor(lambdas, dtype=torch.double)

values_torch = expensive_f_torch(lambdas_torch) 

# Example loss: mean squared amplitude
loss = torch.mean(values_torch)
loss.backward()

print("Loss:", loss.item())
print("dLoss/dtheta:", theta.grad.item())

# Analytic gradient
analytic_loss = 1/3*((theta + 1)**3 - theta**3)
analytic_grad = 2*theta + 1
print("Analytic Loss:", analytic_loss.item())
print("Analytic dLoss/dtheta:", analytic_grad.item())

Loss: 2.4922902960526314
dLoss/dtheta: 3.0921052631578947
Analytic Loss: 2.333333333333333
Analytic dLoss/dtheta: 3.0


### My functional version

In [ ]:
import numpy as np
import torch
import adaptive

def mean_expensive_f(theta: torch.Tensor) -> torch.Tensor:
    def expensive_f_torch(lam: torch.Tensor) -> torch.Tensor:
        return torch.pow(lam + theta, 2)
    
    expensive_f_numpy = lambda lam: expensive_f_torch(lam).item()

    learner = adaptive.Learner1D(expensive_f_numpy, bounds=(0.0, 1.0))
    adaptive.runner.simple(learner, loss_goal=0.1)
    
    f_data = learner.to_numpy()
    lambdas_torch = torch.tensor(f_data[:,0], dtype=torch.double)
    values_torch = expensive_f_torch(lambdas_torch) 

    loss = torch.mean(values_torch)
    return loss

mean_expensive_f_grad = torch.func.jacrev(mean_expensive_f)

theta = torch.tensor(1., dtype=torch.double)
print("Loss:", mean_expensive_f(theta).item())
print("dLoss/dtheta:", mean_expensive_f_grad(theta).item())

# Analytic gradient
analytic_loss = 1/3*((theta + 1)**3 - theta**3)
analytic_grad = 2*theta + 1
print("Analytic Loss:", analytic_loss.item())
print("Analytic dLoss/dtheta:", analytic_grad.item())